# 01 — Collecte des organisations de Marrakech

Ce notebook collecte des organisations publiques et privées via Google Places API (New). Il conserve les doublons dans le fichier brut, puis produit un fichier propre avec les **15 colonnes utilisées par Marrakech Career**.

> Avant d'exécuter : activez Places API (New), la facturation Google, puis définissez la variable `GOOGLE_MAP_API`. La clé ne doit jamais être enregistrée dans ce dépôt.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve().parents[1] if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'data' / 'src'))
from companies_pipeline import charger_requetes, collecter_entreprises, nettoyer_entreprises, exporter_excels

CONFIG = ROOT / 'data' / 'config' / 'company_queries.json'
OUTPUTS = ROOT / 'data' / 'outputs'
print('Racine :', ROOT)

Dossier data : C:\Users\hp\Desktop\MarrakechCareer-publish-20261002\data
Historique brut : (1760, 12)
Historique nettoyé : (291, 15)
Google Places activé : False


## Requêtes utilisées

Les requêtes couvrent le secteur public et le secteur privé : informatique, tourisme, hôtellerie, immobilier, BTP, finance, formation, santé, industrie, logistique et commerce. Vous pouvez compléter `data/config/company_queries.json` sans modifier le code.

In [ ]:
requetes = charger_requetes(CONFIG)
print(f'{len(requetes)} requêtes')
requetes[:5]

47 requêtes Google Places configurées
Secteurs : ['Privé', 'Public']


[{'secteur': 'Public', 'requete': 'établissement public Marrakech'},
 {'secteur': 'Public', 'requete': 'agence publique Marrakech'},
 {'secteur': 'Public', 'requete': 'université publique Marrakech'},
 {'secteur': 'Public', 'requete': 'hôpital public Marrakech'},
 {'secteur': 'Public', 'requete': 'office public Marrakech'},
 {'secteur': 'Public', 'requete': 'service régional Marrakech'},
 {'secteur': 'Privé', 'requete': 'entreprise informatique Marrakech'},
 {'secteur': 'Privé', 'requete': 'agence digitale Marrakech'}]

## Collecte Google Places

L'API est appelée avec un *field mask* minimal : nom, adresse, téléphone, site, lien Google Maps, note, avis, catégorie et coordonnées. La pagination, les erreurs réseau et les doublons sont gérés dans le module partagé.

In [ ]:
# Le nombre de pages peut être réduit à 1 pendant les tests afin de maîtriser le coût.
df_brut = collecter_entreprises(requetes, pages_max=3)
print(df_brut.shape)
df_brut.head()

Collecte en ligne ignorée : définissez GOOGLE_MAP_API pour interroger Places.
Historique : 1760 | Google Places : 0 | total brut : 1760


,search_query,place_id,nom,adresse,telephone,site_web,google_maps,note,nombre_avis,categorie,latitude,longitude
0,entreprise intelligence artificielle Marrakech,ChIJwWQgn3bvrw0Rd4BO-FsEmhU,Artificial.ma - Agence Intelligence Artificielle,Marrakech 40000,06 43 73 01 58,NaN,https://maps.google.com/?cid=15565614142643569...,NaN,NaN,Services,31.633838,-7.985246
1,entreprise intelligence artificielle Marrakech,ChIJf_2_LkLtrw0RtBErLCQ5taI,HIGH INTELLIGENCE,"Unnamed Road, Marrakech 40000",05 22 98 01 96,http://www.high-intelligence.ma/,https://maps.google.com/?cid=11724340032451580...,5.0,1.0,Siège social,31.642654,-8.019593
2,agence intelligence artificielle Marrakech,ChIJwWQgn3bvrw0Rd4BO-FsEmhU,Artificial.ma - Agence Intelligence Artificielle,Marrakech 40000,06 43 73 01 58,NaN,https://maps.google.com/?cid=15565614142643569...,NaN,NaN,Services,31.633838,-7.985246
3,AI agency Marrakech,ChIJYzAoNwDvrw0RG5if0inDhn4,@agency@famly,"Jam alfna, Marrakech 40000",06 35 56 32 60,https://www.instagram.com/excursion_3439?igsh=...,https://maps.google.com/?cid=91171890800528978...,5.0,21.0,NaN,31.622522,-7.989826
4,AI agency Marrakech,ChIJwWQgn3bvrw0Rd4BO-FsEmhU,Artificial.ma - Agence Intelligence Artificielle,Marrakech 40000,06 43 73 01 58,NaN,https://maps.google.com/?cid=15565614142643569...,NaN,NaN,Services,31.633838,-7.985246


## Nettoyage et classification

Le nettoyage retire les doublons par `place_id`, conserve Marrakech à partir de l'adresse ou des coordonnées, normalise les champs et attribue domaine, sous-domaine et priorité. Le résultat conserve exactement les colonnes de l'application.

In [ ]:
df_propre = nettoyer_entreprises(df_brut)
print(df_propre.shape)
print(df_propre.columns.tolist())
df_propre.head()

(900, 15)
['nom', 'domaine', 'sous_domaine', 'categorie', 'telephone', 'email', 'site_web', 'adresse', 'latitude', 'longitude', 'note', 'nombre_avis', 'priorite', 'score_priorite', 'google_maps']


,nom,domaine,sous_domaine,categorie,telephone,email,site_web,adresse,latitude,longitude,note,nombre_avis,priorite,score_priorite,google_maps
0,212 Services | Services Informatique,IT & Digital,Autre IT,Consultant,06 70 70 79 83,contact@212-services.com,https://212-services.com/,"N° 2 ETAGE 2 BLOC 21, Marrakech 40000",31.631397,-8.040655,5.0,18.0,Haute,11,https://maps.google.com/?cid=12157903107002902...
1,Chifaa Conseil : Domiciliation d'entreprise a ...,Services aux entreprises,-,Consultant,06 66 94 86 96,direction.chifaaconseil@gmail.com,https://chifaaconseil.com/,Zenith Business Center Rue Mouslim Lot. Boukar...,31.640020,-7.998464,4.9,21.0,Haute,11,https://maps.google.com/?cid=86306382475181043...
2,Excellent Consulting - Comptable Marrakech - F...,Services aux entreprises,-,Consultant,06 73 22 71 20,gestionentreprises21@gmail.com,http://www.fid-ec.com/,"LOT AL MASSAR N°416 2 EME ETAGE, APPT 2, Marra...",31.674944,-8.037470,4.9,116.0,Haute,11,https://maps.google.com/?cid=12782511065637509...
3,Kap Conseils,Services aux entreprises,-,Consultant,07 00 77 01 64,kapconseil@gmail.com,https://www.kap-conseil.com/,"Angle yacoub al mansour imm al amira 3 , 2em E...",31.638212,-8.010131,4.8,25.0,Haute,11,https://maps.google.com/?cid=49068892120538181...
4,"N.Synergy Bureau d'Étude Technique, Développem...",IT & Digital,IA & Data,Consultant,06 88 21 22 23,jel.synergy@gmail.com,https://www.nsynergy.ma/,"4eme Etage N° 17, Centre d'affaires JOUDAD, N9...",31.656433,-8.017697,4.6,7.0,Haute,11,https://maps.google.com/?cid=58169760936732916...


In [ ]:
avant, apres = exporter_excels(df_brut, df_propre, OUTPUTS)
print('Créé :', avant)
print('Créé :', apres)
print('Brut:', len(df_brut), '| unique propre:', len(df_propre))

Créé : C:\Users\hp\Desktop\MarrakechCareer-publish-20261002\data\outputs\entreprises_avant_nettoyage.xlsx
Créé : C:\Users\hp\Desktop\MarrakechCareer-publish-20261002\data\outputs\entreprises_apres_nettoyage.xlsx
Brut: 1760 | unique propre: 900


## Contrôles qualité

À vérifier avant import dans Supabase : le nombre de `place_id` uniques, les coordonnées manquantes, la couverture des domaines et les organisations classées `Non classé`.

In [ ]:
print('Doublons bruts :', df_brut.duplicated('place_id').sum())
print('Notes manquantes :', df_propre['note'].isna().sum())
print(df_propre['domaine'].value_counts())